In [5]:
import sys
import argparse
import logging
import json
import pandas as pd
import numpy as np
import h5py
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional
import re
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

from src.util.log_manager import setup_logging
from src.simulation.run_emates import run_parallel_emates_simulations
from src.util.path_manager import get_paths


In [6]:
class EMATESPipeline:
    """eMATESシミュレーションの統合パイプライン - データ整理特化版"""
    
    def __init__(self, parallel_workers: int = 4, debug: bool = False):
        self.parallel_workers = parallel_workers
        self.debug = debug
        self.logger = self._setup_logging()
        self.scenario_data = {}
        self.optimization_results = {}  # 最適化結果保存用

                
    # ===== 単年度逐次配置計画 =====
    def run_yearly_optimization(self, scenario_id: int, year: int, worker_ids: List[int] = [1]) -> Dict:
        """単年度の最適化実行"""
        self.logger.info(f"シナリオ{scenario_id} - {year}年目の最適化開始")
        
        # 1. eMATES実行（現在のCS設定で）
        yearly_data = self.collect_scenario_data(scenario_id, year, worker_ids)
        
        # 2. データ評価
        performance = self._evaluate_performance(yearly_data)
        
        # 3. MILP問題定式化・最適化
        optimal_config = self._optimize_cs_configuration(yearly_data, performance)
        
        # 4. 次年度設定更新
        self._update_cs_configuration(scenario_id, year + 1, optimal_config)
        
        # 結果保存
        result = {
            "scenario_id": scenario_id,
            "year": year,
            "performance": performance,
            "optimal_config": optimal_config,
            "simulation_data": yearly_data
        }
        
        key = f"scenario_{scenario_id}_year_{year}"
        self.optimization_results[key] = result
        
        self.logger.info(f"シナリオ{scenario_id} - {year}年目の最適化完了")
        return result
    
    # ===== 長期間シナリオ計画 =====
    def run_long_term_scenario_planning(self, scenario_ids: List[int] = [1, 2, 3, 4], 
                                      start_year: int = 0, end_year: int = 10) -> Dict:
        """複数シナリオの長期間計画実行"""
        self.logger.info(f"長期間シナリオ計画開始: シナリオ{scenario_ids}, {start_year}-{end_year}年")
        
        all_results = {}
        
        for scenario_id in scenario_ids:
            self.logger.info(f"シナリオ{scenario_id}の処理開始")
            scenario_results = {}
            
            # 初期CS設定
            self._initialize_cs_configuration(scenario_id)
            
            # 年次ループ
            for year in range(start_year, end_year + 1):
                yearly_result = self.run_yearly_optimization(scenario_id, year)
                scenario_results[f"year_{year}"] = yearly_result
                
                # 進捗表示
                progress = (year - start_year + 1) / (end_year - start_year + 1) * 100
                self.logger.info(f"シナリオ{scenario_id}: {progress:.1f}% 完了")
            
            all_results[f"scenario_{scenario_id}"] = scenario_results
            self.logger.info(f"シナリオ{scenario_id}完了")
        
        self.logger.info("全シナリオの長期間計画完了")
        return all_results
    
    # ===== コア機能 =====
    def collect_scenario_data(self, scenario_id: int, year: int, worker_ids: List[int]) -> Dict:
        """シナリオ・年度別データ収集"""
        from src.util.scenario import get_adoption_rate
        
        ev_rate = get_adoption_rate(scenario_id, year)
        
        scenario_data = {
            "scenario_id": scenario_id,
            "year": year,
            "metadata": {
                "ev_adoption_rate": ev_rate,
                "collected_at": datetime.now().isoformat()
            },
            "cs_data": {}
        }
        
        for worker_id in worker_ids:
            try:
                paths = get_paths(worker_id)
                result_dir = Path(paths["result"])
                worker_data = self._load_worker_data(worker_id, result_dir)
                scenario_data["cs_data"][f"worker_{worker_id}"] = worker_data
            except Exception as e:
                self.logger.error(f"Worker {worker_id} 処理エラー: {e}")
        
        return scenario_data
    
    def _evaluate_performance(self, yearly_data: Dict) -> Dict:
        """性能評価"""
        metrics = {
            "profit": 0,
            "waiting_time": 0,
            "loss_count": 0,
            "utilization": 0
        }
        
        for worker_name, data in yearly_data["cs_data"].items():
            # 利潤計算
            revenue = self._calculate_revenue(data)
            initial_cost = self._initial_cost(data)
            operating_cost = self._operating_cost(data['timeseries'])
            profit = revenue - initial_cost - operating_cost
            
            # 待機時間計算
            vehicle_trip = data['vehicle_trip']
            non_zero = vehicle_trip[vehicle_trip['WaitingEntryTime'] != 0]
            if not non_zero.empty:
                waiting_times = non_zero['startChargingTime'].values - non_zero['WaitingEntryTime'].values
                avg_waiting = waiting_times[waiting_times > 0].mean()
            else:
                avg_waiting = 0
            
            # ロス回数
            loss_count = len(data['charging_loss'])
            
            metrics["profit"] += profit
            metrics["waiting_time"] += avg_waiting
            metrics["loss_count"] += loss_count
        
        return metrics
    
    def _optimize_cs_configuration(self, yearly_data: Dict, performance: Dict) -> Dict:
        """MILP最適化（簡略版）"""
        # 現在の設定を基準に改善案を生成
        current_config = yearly_data["cs_data"]["worker_1"]
        
        # 簡単な最適化ロジック（実際にはMILPソルバーを使用）
        if performance["waiting_time"] > 100:  # 待機時間が長い場合
            # ポート数を増加
            new_ports = [p + 1 for p in current_config["ports"]]
        else:
            # 現状維持
            new_ports = current_config["ports"]
        
        optimal_config = {
            "csids": current_config["csids"],
            "ports": new_ports,
            "cap_kw": current_config["cap_kw"]
        }
        
        return optimal_config
    
    def _update_cs_configuration(self, scenario_id: int, year: int, config: Dict):
        """CS設定更新（次年度用）"""
        # 実際にはeMATESの設定ファイルを更新
        self.logger.info(f"シナリオ{scenario_id} - {year}年目のCS設定更新")
        # TODO: csList.txtファイルの更新処理
    
    def _initialize_cs_configuration(self, scenario_id: int):
        """初期CS設定"""
        self.logger.info(f"シナリオ{scenario_id}の初期CS設定")
        # TODO: 初期設定の定義
    
    # ===== 結果分析・可視化 =====
    def get_optimization_summary(self) -> pd.DataFrame:
        """最適化結果のサマリー取得"""
        summary_data = []
        
        for key, result in self.optimization_results.items():
            summary_data.append({
                "scenario_id": result["scenario_id"],
                "year": result["year"],
                "profit": result["performance"]["profit"],
                "waiting_time": result["performance"]["waiting_time"],
                "loss_count": result["performance"]["loss_count"],
                "total_ports": sum(result["optimal_config"]["ports"])
            })
        
        return pd.DataFrame(summary_data)
    
    def save_results(self, filepath: str = None):
        """結果保存"""
        if filepath is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filepath = f"optimization_results_{timestamp}.json"
        
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(self.optimization_results, f, ensure_ascii=False, indent=2, default=str)
        
        self.logger.info(f"最適化結果を保存: {filepath}")
        
    def _setup_logging(self):
        """ログ設定"""
        setup_logging(self.debug)
        return logging.getLogger(__name__)
    
    def load_cs_configurations(self, config_path: str) -> List[Dict]:
        """CS配置設定を読み込み"""
        try:
            with open(config_path, 'r') as f:
                solutions = json.load(f)
            self.logger.info(f"CS配置設定を読み込み: {len(solutions)}個の設定")
            return solutions
        except (ValueError, FileNotFoundError) as e:
            self.logger.error(f"設定読み込みエラー: {e}")
            raise
    
    def run_simulations(self, solutions: List[Dict]) -> int:
        """シミュレーション実行"""
        self.logger.info(f"シミュレーション開始: {len(solutions)}個の設定")
        
        # 各設定を順次実行（並列化も可能）
        for i, solution in enumerate(solutions):
            self.logger.info(f"設定 {i+1}/{len(solutions)} を実行中...")
            result = run_parallel_emates_simulations(solution, self.parallel_workers)
            if result != 0:
                self.logger.error(f"設定 {i+1} の実行に失敗")
                return result
        
        self.logger.info("全シミュレーション完了")
        return 0
    
    def _load_T_files(self, emates_dir: Path, start_datetime: str = "2024-01-01 00:00:00") -> pd.DataFrame:
        """Tファイル（時系列データ）の読み込み - datetime対応版"""
        try:
            t_files = sorted(
                emates_dir.glob("T*.csv"),
                key=lambda x: int(re.search(r"T(\d{6})", x.stem).group(1))
            )
            
            if not t_files:
                return pd.DataFrame()
            
            t_dfs = []
            for f in t_files:
                elapsed = int(re.search(r"T(\d{6})", f.stem).group(1))
                df = pd.read_csv(f)
                df['ElapsedTime'] = elapsed
                t_dfs.append(df)
            
            # データフレーム結合
            combined_df = pd.concat(t_dfs, ignore_index=True)
            
            # CSIDごとにCap_kWとwaitingLineを集約
            unified_df = self._aggregate_cs_data_unified(combined_df)
            
            # 分離版でもdatetime変換を適用
            unified_df = self._add_datetime_index(unified_df, start_datetime)
            
            return unified_df
                
        except Exception as e:
            self.logger.warning(f"Tファイル読み込みエラー: {e}")
            return pd.DataFrame()

    def _add_datetime_index(self, df: pd.DataFrame, start_datetime: str) -> pd.DataFrame:
        """DataFrameにdatetimeインデックスを追加"""
        if df.empty or 'ElapsedTime' not in df.columns:
            return df
        
        try:
            # 開始時刻を基準にしたdatetimeインデックス作成
            start_time = pd.to_datetime(start_datetime)
            
            # ElapsedTimeを秒単位と仮定してTimedeltaに変換
            df['Timestamp'] = start_time + pd.to_timedelta(df['ElapsedTime'], unit='s')
            
            # Timestampをインデックスに設定
            df = df.set_index('Timestamp')
            df = df.sort_index()
            
            self.logger.info(f"datetime変換完了: {len(df)}行, 期間: {df.index.min()} - {df.index.max()}")
            
            return df
            
        except Exception as e:
            self.logger.warning(f"datetime変換エラー: {e}")
            return df
    
    def _aggregate_cs_data_unified(self, df: pd.DataFrame) -> pd.DataFrame:
        """CSIDごとのデータを時間ごとに集約（分離版）"""
        try:
            # Cap_kWとwaitingLineを同じDataFrameとして作成
            cap_kw_df = df.pivot_table(
                index='ElapsedTime',
                columns='Csid',
                values='Cap_kW',
                fill_value=0
            )
            cap_kw_df.columns = [f'{csid}' for csid in cap_kw_df.columns]
            
            waiting_df = df.pivot_table(
                index='ElapsedTime',
                columns='Csid',
                values='waitingLine',
                fill_value=0
            )
            waiting_df.columns = [f'{csid}' for csid in waiting_df.columns]
                        
            # ElapsedTimeを列として復元
            unified_df = pd.concat([cap_kw_df, waiting_df], axis=1)
            unified_df.reset_index(inplace=True)
            
            self.logger.info(f"CS情報の統一: Cap_kW {unified_df.shape}")
            
            return unified_df
            
        except Exception as e:
            self.logger.warning(f"CS集約エラー: {e}")
            return pd.DataFrame()


    def _resample_timeseries(self, df: pd.DataFrame, freq: str = '1H') -> pd.DataFrame:
        """時系列データのリサンプリング（補間用）"""
        if df.empty or not isinstance(df.index, pd.DatetimeIndex):
            return df
        
        try:
            # 数値カラムのみを対象にリサンプリング
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            
            if len(numeric_cols) == 0:
                return df
            
            # 数値データを平均でリサンプリング
            resampled = df[numeric_cols].resample(freq).mean()
            
            # 非数値データは最初の値で前方補完
            categorical_cols = df.select_dtypes(exclude=[np.number]).columns
            if len(categorical_cols) > 0:
                categorical_resampled = df[categorical_cols].resample(freq).first()
                resampled = pd.concat([resampled, categorical_resampled], axis=1)
            
            # 欠損値の補間
            resampled = resampled.interpolate(method='linear')
            
            self.logger.info(f"リサンプリング完了: {freq} 間隔, {len(resampled)}行")
            return resampled
            
        except Exception as e:
            self.logger.warning(f"リサンプリングエラー: {e}")
            return df

    def _load_worker_data(self, worker_id: int, result_dir: Path, 
                        start_datetime: str = "2024-01-01 00:00:00",
                        resample_freq: Optional[str] = None) -> Dict:
        """単一ワーカーのデータ読み込み - datetime対応版"""
        emates_dir = result_dir / "emates"
        
        # 時系列データ（Tファイル）- datetime変換
        timeseries_df = self._load_T_files(emates_dir, start_datetime)
        
        # 必要に応じてリサンプリング
        if resample_freq:
            t_data = self._resample_timeseries(t_data.get('cap_kw'), resample_freq)
            
        
        # 充電ロスデータ
        charging_loss = self._load_charging_loss(result_dir)
        # 走行データ
        vehicle_trip = self._load_vehicle_trip(result_dir)
        # CSIDとポート情報の抽出
        cs_data = self._get_cs_list(worker_id)
        
        return {
            "worker_id": worker_id,
            "result_dir": result_dir,
            "timeseries": timeseries_df,
            "charging_loss": charging_loss,
            "vehicle_trip": vehicle_trip,
            "csids": cs_data['csids'],
            "ports": cs_data['ports'],
            "cap_kw": cs_data['cap_kw'],
            "total_ports": cs_data['total_ports'],
        }
    
    def _get_cs_list(self, worker_id: int) -> Dict[str, any]:
        """CSリストファイルからCS情報を取得"""
        try:
            paths = get_paths(worker_id)
            csList_file = paths["csList"]
            cs_info = pd.read_csv(csList_file, sep=',', header=None, names=['CSID', 'Port', 'Cap_kw'])
            
            # CS情報を辞書形式で整理
            cs_data = {
                'csids': cs_info['CSID'].tolist(),
                'ports': cs_info['Port'].tolist(),
                'cap_kw': cs_info['Cap_kw'].tolist(),
                'total_ports': cs_info['Port'].sum(),
                'total_cs_count': len(cs_info)
            }
            
            self.logger.info(f"Worker {worker_id}: CS情報取得完了 - {cs_data['total_cs_count']}箇所, 総ポート数: {cs_data['total_ports']}")
            return cs_data
            
        except Exception as e:
            self.logger.warning(f"Worker {worker_id} CS情報取得エラー: {e}")
            return {
                'csids': [],
                'ports': [],
                'cap_kw': [],
                'total_ports': 0,
                'total_cs_count': 0,
            }
        
    def collect_worker_data(self, max_workers: int = 100, 
                           start_datetime: str = "2024-01-01 00:00:00",
                           resample_freq: Optional[str] = None) -> Dict[str, Dict]:
        """ワーカーデータの収集・整理のみ実行"""
        self.logger.info("データ収集・整理を開始（datetime対応）")
        
        worker_data_collection = {}
        
        for worker_id in range(1, max_workers + 1):
            try:
                paths = get_paths(worker_id)
                result_dir = Path(paths["result"])
                
                if not result_dir.exists():
                    self.logger.info(f"Worker {worker_id} が見つからないため処理終了")
                    break
                
                # データ収集（datetime対応）
                worker_data = self._load_worker_data(
                    worker_id, result_dir, start_datetime, resample_freq
                )
                
                worker_data_collection[f"worker_{worker_id}"] = worker_data
                
            except Exception as e:
                self.logger.error(f"Worker {worker_id} データ収集エラー: {e}")
                continue
        
        self.logger.info(f"総計 {len(worker_data_collection)} ワーカーのデータを収集")
        return worker_data_collection
    
    def _load_charging_loss(self, result_dir: Path) -> pd.DataFrame:
        """充電ロスデータの読み込み"""
        try:
            charging_loss_path = result_dir / "chargingLoss.txt"
            if not charging_loss_path.exists():
                return pd.DataFrame()
            
            df = pd.read_csv(
                charging_loss_path, sep=',',header=None,
                names=['Time', 'EVID', 'CSID', 'WaitingNum', 'NumPorts', 'SOC']
            )
            # 時間をmsから秒に変換
            df['Time_sec'] = df['Time'] / 1000
            
            # 60秒間隔にビニング
            df['ElapsedTime'] = (df['Time_sec'] // 60) * 60
            df.drop(columns=['Time_sec','Time'], inplace=True)
            df = self._add_datetime_index(df, "2024-01-01 00:00:00")
            return df
        
        except Exception as e:
            self.logger.warning(f"充電ロスデータ読み込みエラー: {e}")
            return pd.DataFrame()
    
    def _load_vehicle_trip(self, result_dir: Path) -> pd.DataFrame:
        """走行データの読み込み"""
        try:
            vehicle_trip_path = result_dir / "vehicleTrip.txt"
            if not vehicle_trip_path.exists():
                return pd.DataFrame()
            
            df = pd.read_csv(
                vehicle_trip_path, sep=r',',usecols=[0, 2, 3, 4, 5, 8,9,10,11, 14],
                names=['EVID','StartTime', 'EndTime', 'WaitingEntryTime','startChargingTime',
                       'startID','goalID','tripLength','CSID', 'InitialSOC'],
                dtype=str
            )
                    # 特殊文字の処理
            def clean_numeric_value(value):
                """数値変換前の前処理"""
                if pd.isna(value) or value == '' or value == '******':
                    return np.nan
                try:
                    return float(value)
                except (ValueError, TypeError):
                    return np.nan
            
            # 各列を適切な型に変換
            numeric_columns = ['EVID', 'StartTime', 'EndTime', 'WaitingEntryTime', 
                            'startChargingTime', 'startID', 'goalID', 'tripLength', 
                            'CSID', 'InitialSOC']
            
            for col in numeric_columns:
                if col in df.columns:
                    df[col] = df[col].apply(clean_numeric_value)
            # 時間を秒に変換
            
            df['StartTime'] = df['StartTime'] / 1000
            df['EndTime'] = df['EndTime'] / 1000
            df['WaitingEntryTime'] = df['WaitingEntryTime'] / 1000
            df['startChargingTime'] = df['startChargingTime'] / 1000
            
            return df
            
        except Exception as e:
            self.logger.warning(f"走行データ読み込みエラー: {e}")
            return pd.DataFrame()
    
    def run_data_collection_pipeline(self, config_path: Optional[str] = None) -> Dict[str, Dict]:
        """データ収集パイプライン実行"""
        try:
            # シミュレーション実行（設定ファイルが指定された場合のみ）
            if config_path:
                solutions = self.load_cs_configurations(config_path)
                sim_result = self.run_simulations(solutions)
                if sim_result != 0:
                    raise RuntimeError("シミュレーション実行に失敗")
            
            # データ収集と整理
            worker_data = self.collect_worker_data()
            
            if worker_data:
                self.logger.info("データ収集パイプライン完了")
                return worker_data
            else:
                self.logger.warning("収集されたデータが空です")
                return {}
            
        except Exception as e:
            self.logger.error(f"データ収集パイプライン実行エラー: {e}")
            return {}

In [7]:
pipeline = EMATESPipeline(parallel_workers=4, debug=True)
result_year_dict = pipeline.run_yearly_optimization(scenario_id=1, year=0, worker_ids=[1])

2025-06-09 22:13:27,548 - INFO - ログ記録を開始しました: C:\Users\echiz\00_研究コード\eMATES解析_GA\emates\src\util\logs\simulation_20250609_221327.log
2025-06-09 22:13:27,548 - INFO - シナリオ1 - 0年目の最適化開始


ImportError: cannot import name 'get_adoption_rate' from 'src.util.scenario' (c:\Users\echiz\00_研究コード\eMATES解析_GA\emates\src\util\scenario.py)

In [ ]:
# パイプライン実行後、そのまま分析
pipeline = EMATESPipeline()
df = pipeline.collect_scenario_data(scenario_id=1, worker_ids=[1])



2025-06-09 19:04:49,931 - INFO - ログ記録を開始しました: C:\Users\echiz\00_研究コード\eMATES解析_GA\emates\src\util\logs\simulation_20250609_190449.log
2025-06-09 19:04:54,194 - INFO - CS情報の統一: Cap_kW (1440, 23)
2025-06-09 19:04:54,194 - INFO - datetime変換完了: 1440行, 期間: 2024-01-01 00:01:00 - 2024-01-02 00:00:00
2025-06-09 19:04:54,213 - INFO - datetime変換完了: 6行, 期間: 2024-01-01 08:37:00 - 2024-01-01 10:09:00
2025-06-09 19:04:54,260 - INFO - Worker 1: CS情報取得完了 - 9箇所, 総ポート数: 11


In [ ]:
# 辞書の中身を確認する基本的な方法
print(list(df.keys()))  # リスト形式で表示
print(df.get('scenario_id'))  # キーが存在しない場合はNoneを返す
# 3. 階層構造の可視化
import json
print(json.dumps(df, indent=2, default=str))  # 日付などもstr変換


['plan_id', 'metadata', 'cs_data']
1
{
  "plan_id": 1,
  "metadata": {
    "created_at": "2025-06-09 19:04:49.932945",
    "worker_count": 1
  },
  "cs_data": {
    "worker_1": {
      "worker_id": 1,
      "result_dir": "\\\\wsl.localhost\\ubuntu-22.04\\home\\tsato-cnlab\\Emates\\eMATES_2308\\network\\coupled_network_shikata_1\\result",
      "timeseries": "                     ElapsedTime  90000000  90000001  90000200  90000201  \\\nTimestamp                                                                  \n2024-01-01 00:01:00           60       0.0       0.0       0.0       0.0   \n2024-01-01 00:02:00          120       0.0       0.0       0.0       0.0   \n2024-01-01 00:03:00          180      90.0       0.0       0.0       0.0   \n2024-01-01 00:04:00          240      90.0       0.0       0.0       0.0   \n2024-01-01 00:05:00          300      90.0       0.0       0.0       0.0   \n...                          ...       ...       ...       ...       ...   \n2024-01-01 23:56:00   

In [29]:
# 直接評価・分析
for worker_name, data in worker_data.items():
    timeseries = data['timeseries']
    charging_loss = data['charging_loss']
    vehicle_trip = data['vehicle_trip']
    
    # 評価指標計算
    total_capacity = timeseries['cap_kw'].sum().sum()
    non_zero = vehicle_trip[vehicle_trip['WaitingEntryTime'] != 0]
    waiting_times = non_zero['startChargingTime'].values - non_zero['WaitingEntryTime'].values

    
    # 待機時間の統計
    avg_waiting = waiting_times.mean()
    max_waiting = waiting_times.max()
    min_waiting = waiting_times.min()
    total_waiting_events = len(waiting_times[waiting_times > 0])  # 待機が発生した回数

    
    loss_count = len(charging_loss) if not charging_loss.empty else 0
    
    print(f"{worker_name}:")
    print(f"  容量: {total_capacity:.2f}kWh")
    print(f"  平均待機時間: {avg_waiting:.2f}ms")
    print(f"  最大待機時間: {max_waiting:.2f}ms") 
    print(f"  待機発生回数: {total_waiting_events}回")
    print(f"  充電ロス: {loss_count}回")
    print("-" * 40)

worker_1:
  容量: 62453970.00kWh
  平均待機時間: 22862.96ms
  最大待機時間: 425700.00ms
  待機発生回数: 2回
  充電ロス: 6回
----------------------------------------
worker_2:
  容量: 62444700.00kWh
  平均待機時間: 227735.56ms
  最大待機時間: 3931700.00ms
  待機発生回数: 12回
  充電ロス: 25回
----------------------------------------


In [51]:
18.17e-04

0.001817

In [ ]:
def objective()-> float:
    """目的関数の定義
    事業者の利潤最大化を目的とする"""
    init_cost = _initial_cost()
    op_cost = _operating_cost()
    
    benefit = 
    return benefit

def _initial_cost(cs_configuration: List[Dict])-> float:
    """充電器の初期費用計算
    
    Args:
        cs_configuration: CS配置設定のリスト
                         [{'id': CS_ID, 'capacity': 充電容量(kW), 'location': (x, y)}, ...]
    
    Returns:
        float: 初期費用総額（万円）
    """
    charger_cost = {'50kW': 380, '90kW':670,'100kW': 730} # kWあたりのコスト：万円
    installation_cost = 0 # 設置コストは未定義
    substation_per_kW = 2 # 変電所のkWあたりのコスト：万円/kW
    
    total_charger_cost = 0      # 充電器コスト
    total_installation_cost = 0 # 設置コスト
    total_substation_cost = 0          # 総容量
    
    for cs in cs_configuration:
        capacity = cs.get('capacity', 50)  # デフォルト50kW
        num_ports = cs.get('num_ports', 0)  # ポート数（デフォルト0）
        
        
        # 充電器種別を判定
        if capacity <= 50:
            charger_type = '50kW'
        elif capacity <= 90:
            charger_type = '90kW'
        else:
            charger_type = '100kW'
        
        # 充電器コスト計算
        unit_cost = charger_cost[charger_type]
        total_charger_cost += unit_cost
        
        # 設置コスト計算
        total_installation_cost += installation_cost
        
        # 総容量累計
        total_capacity += capacity
    
    # 変電所コスト計算
    substation_cost = total_capacity * substation_per_kW
    
    # 初期費用合計
    initial_cost = total_charger_cost + total_installation_cost + substation_cost
    
    return initial_cost

def _operating_cost(df_cap_kw: pd.DatFrame, operation_months: int = 12)-> float:
    """運用コスト計算
    
    Args:
        df_cap_kw: 一日の時系列データ（DataFrame）
        operation_months: 運用期間（月数）
    
    Returns:
        float: 運用コスト（万円）
    """
    electricity_basic_fee = 1911.8e-04  # 基本料金：万円/kW・月
    electricity_use_fee = 18.07e-04     # 従量料金：万円/kWh
    
    # 一日の総充電量（kWh）を計算
    total_energy_kwh = df_cap_kw.sum().sum() /60  # 1日の総容量をkWhに変換
    max_capacity = df_cap_kw.max()  # 最大容量（kW）
    
    # 基本料金（容量分）
    basic_cost = electricity_basic_fee * max_capacity * operation_months
    # 従量料金（使用電力分）
    usage_cost = electricity_use_fee * total_energy_kwh * operation_months * 365  # 年間の使用電力分
    
    # 運用コスト合計
    operating_cost = basic_cost + usage_cost
    
    return operating_cost


In [52]:
cs_config = {
    "csid": [1,2,3],
    "charger_type": ["50kW","90kW","100kW"],
    "num_ports": [2, 1, 1]
}

In [19]:
non_zero = vehicle_trip[vehicle_trip['WaitingEntryTime'] != 0]
non_zero.head(1000)

,EVID,StartTime,EndTime,WaitingEntryTime,startChargingTime,startID,goalID,tripLength,CSID,InitialSOC
159,7,21800,1742100,159100,159100,5001,1437,5447.07,900000.0,0.2
481,1181,3199100,4963100,3379100,3379100,5001,1437,5447.09,900000.0,0.2
967,2897,12808900,14519300,12955100,12955100,5001,169,5472.08,900000.0,0.2
1545,4480,22723100,24432900,22868700,22868700,5001,169,5472.09,900000.0,0.2
2687,6904,26488200,28882000,26765700,26765700,5003,1123,9112.89,900002.0,0.2
...,...,...,...,...,...,...,...,...,...,...
19286,63377,81610400,83911800,81811100,81811100,5005,1031,9099.73,900006.0,0.2
19314,63881,82313600,84091500,82525800,82525800,5001,169,5472.96,900000.0,0.2
19419,64252,82825100,84745600,83060400,83060400,5003,169,6448.70,900002.0,0.2
19642,64961,84305200,******,84456500,84456500,5001,1123,5709.22,0.0,NaN
